In [6]:
from pathlib import Path
import pandas as pd
import numpy as np

In [7]:
ROOT = Path('/Users/zhu/Desktop/实习项目/盈远数据')
OUT = ROOT / 'merged_data'
OUT.mkdir(exist_ok=True)

In [8]:
PRICE_COLS = ['last', 'open', 'high', 'low'] + \
             [f'{s}_price{i}' for s in ('ask', 'bid') for i in range(1, 6)]
VOL_COLS = [f'{s}_volume{i}' for s in ('ask', 'bid') for i in range(1, 6)]
KEEP=['trade_time', 'code', 'pre_close', 'last', 'open', 'high', 'low',
        'volume', 'amount', 'num_trades'] + \
       [c for c in PRICE_COLS if '_price' in c] + VOL_COLS


In [9]:
def load_raw(folder):
    """读取一个标的文件夹下所有月度快照并纵向拼接"""
    files=sorted(Path(folder).glob('*_snapshot.csv'))
    df=pd.concat([pd.read_csv(f, index_col=0, encoding='utf-8-sig') for f in files], ignore_index=True)
    print(f'{Path(folder).name}: {len(files)} 个文件, 原始 {len(df):,} 行')
    return df

def clean(df, start, end, max_dev=0.10):
    """清洗单个标的,返回 (清洗后的数据, 每一步的行数报告)"""
    rep = {'raw': len(df)}
    df = df.copy()
    # 1) 统一时间戳,并截取目标日期区间
    df['trade_time'] = pd.to_datetime(df['trade_time'], errors='coerce')
    df=df.dropna(subset=['trade_time'])
    end_excl = pd.Timestamp(end) + pd.Timedelta(days=1)
    df=df[(df['trade_time'] >= start) & (df['trade_time'] < end_excl)]
    rep['in_range']=len(df)
    # 2) 排序+去重
    df=df.sort_values('trade_time').drop_duplicates('trade_time', keep='last')
    rep['dedup'] = len(df)
    # 3) 只保留连续竞价时段 09:30-11:30, 13:00-15:00
    t = df['trade_time'].dt.time
    am = (t >= pd.Timestamp('09:30').time()) & (t <= pd.Timestamp('11:30').time())
    pm = (t >= pd.Timestamp('13:00').time()) & (t <= pd.Timestamp('15:00').time())
    df = df[(am | pm) & df['trading_phase_code'].str.startswith('T')]
    rep['continuous'] = len(df)
    df = df[KEEP].copy()

    # 4)  缺失值:价格 0 -> NaN
    df[PRICE_COLS] = df[PRICE_COLS].where(df[PRICE_COLS] > 0)

    # 5) 异常值：1. 最新价偏离昨日收盘价超过 max_dev（涨停或跌停）；2.买一高于卖一
    bad=(df['last'] / df['pre_close'] - 1).abs()> max_dev
    df.loc[bad,'last']=np.nan
    crossed=df['bid_price1']>df['ask_price1']
    df.loc[crossed,['bid_price1', 'ask_price1']]= np.nan
    rep['outlier_last']=int(bad.sum())
    rep['crossed']=int(crossed.sum())

    # 6) 日内前向填充:只填成交类价格;盘口空档保留 NaN(代表该档无挂单,是真实的流动性状态)
    FILL_COLS = ['last', 'open', 'high', 'low']
    rep['nan_last']=int(df['last'].isna().sum())
    day = df['trade_time'].dt.date
    df[FILL_COLS] = df.groupby(day)[FILL_COLS].ffill()
    df = df.dropna(subset=['last'])
    # 7) 汇总:最终行数、交易日数、买一/卖一为空的比例(%)
    rep['final'] = len(df)
    rep['days'] = df['trade_time'].dt.date.nunique()
    rep['bid1_nan_pct'] = round(df['bid_price1'].isna().mean() * 100, 3)
    rep['ask1_nan_pct'] = round(df['ask_price1'].isna().mean() * 100, 3)

    return df.reset_index(drop=True), rep


In [10]:
TASKS = [
    ('511090', ROOT / '国债ETF数据/3秒快照/511090', '2025-07-01', '2025-12-31'),
    ('511130', ROOT / '国债ETF数据/3秒快照/511130', '2026-01-01', '2026-02-28'),
    ('019742', ROOT / '债券数据/3秒快照/019742',   '2026-01-01', '2026-02-28'),
    ('019776', ROOT / '债券数据/3秒快照/019776',   '2026-01-01', '2026-02-28'),
    ('019789', ROOT / '债券数据/3秒快照/019789',   '2026-01-01', '2026-02-28'),
]

data,reports={},{}
for code,folder, start, end in TASKS:
    df, rep = clean(load_raw(folder), start, end)
    name = f"{code}_{start.replace('-', '')}_{end.replace('-', '')}_merged.csv"
    df.to_csv(OUT / name, index=False, date_format='%Y-%m-%d %H:%M:%S')
    data[code], reports[code] = df, rep

report=pd.DataFrame(reports).T
count_cols = [c for c in report.columns if not c.endswith('_pct')]
report[count_cols] = report[count_cols].astype(int)
report



511090: 6 个文件, 原始 674,843 行
511130: 2 个文件, 原始 180,581 行
019742: 3 个文件, 原始 240,686 行
019776: 3 个文件, 原始 239,442 行
019789: 3 个文件, 原始 242,596 行


,raw,in_range,dedup,continuous,outlier_last,crossed,nan_last,final,days,bid1_nan_pct,ask1_nan_pct
511090,674843,674843,674843,604853,0,0,0,604853,126,0.0,0.0
511130,180581,180581,180581,162921,0,0,0,162921,34,0.0,0.0
019742,240686,190737,189614,160199,0,0,62,160137,34,0.0,0.0
019776,239442,189374,188219,159512,0,0,486,159026,34,0.0,0.0
019789,242596,192066,190668,160426,0,0,1086,159340,34,0.0,0.0
